# vLLM Tutorial 2: Supported Models and Model Serving

## Overview

This notebook covers:

- Comprehensive list of supported models

- Model architecture families

- Model serving options and deployment

- OpenAI-compatible API server

- Advanced configuration parameters

- Quantization options (AWQ, GPTQ, INT8)

- Multi-GPU deployment

---

## Supported Model Architectures

vLLM supports a wide range of model architectures from HuggingFace. Here's a comprehensive list:

### 1. Decoder-Only Models (Causal LM)

These are the most common models for text generation:

#### Meta Models

- **LLaMA** (1, 2, 3, 3.1, 3.2, 3.3): 7B, 8B, 13B, 70B, 405B

- **CodeLlama**: Code-specialized variants

- **OPT**: 125M to 66B parameters

#### Mistral AI Models

- **Mistral**: 7B

- **Mixtral**: 8x7B, 8x22B (Mixture of Experts)

- **Codestral**: Code-specialized

#### Other Popular Models

- **GPT-2**: 124M to 1.5B

- **GPT-J**: 6B

- **GPT-NeoX**: 20B

- **Falcon**: 7B, 40B, 180B

- **MPT**: 7B, 30B

- **Bloom**: 560M to 176B

- **Qwen** (Qwen1.5, Qwen2, Qwen2.5): 0.5B to 72B

- **Yi**: 6B, 34B

- **Phi** (Microsoft): Phi-2, Phi-3

- **StableLM**: Various sizes

- **Baichuan**: 7B, 13B

- **ChatGLM**: 6B

- **Gemma** (Google): 2B, 7B

### 2. Vision-Language Models (Multi-modal)

- **LLaVA**: Image + Text understanding

- **LLaVA-NeXT**: Enhanced vision capabilities

- **Qwen-VL**: Vision-language model

### 3. Code Models

- **CodeLlama**: Meta's code model

- **StarCoder**: BigCode's models

- **WizardCoder**: Code-specialized

- **DeepSeek-Coder**: Advanced code model

---

## Loading Different Models

Let's see how to load various models from HuggingFace.

### Example 1: Loading LLaMA Models

In [ ]:
from vllm import LLM, SamplingParams

# LLaMA 3.2 (3B parameters) - Good for testing
llm = LLM(
    model="meta-llama/Llama-3.2-3B",
    trust_remote_code=True
)

# Test generation
prompt = "Explain machine learning in one sentence:"
sampling_params = SamplingParams(temperature=0.7, max_tokens=50)
output = llm.generate([prompt], sampling_params)[0]

print(f"Model: meta-llama/Llama-3.2-3B")
print(f"Response: {output.outputs[0].text}")

### Example 2: Loading Mistral Models

In [ ]:
# Mistral 7B - Excellent performance for its size
llm = LLM(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    trust_remote_code=True
)

# Mistral uses specific prompt format for instruct models
prompt = "[INST] What is the capital of France? [/INST]"
sampling_params = SamplingParams(temperature=0.1, max_tokens=30)
output = llm.generate([prompt], sampling_params)[0]

print(f"Model: Mistral-7B-Instruct")
print(f"Response: {output.outputs[0].text}")

### Example 3: Loading Mixtral (Mixture of Experts)

In [ ]:
# Mixtral 8x7B - Uses 8 experts, only activates 2 per token
# Requires significant GPU memory (~90GB)
llm = LLM(
    model="mistralai/Mixtral-8x7B-Instruct-v0.1",
    tensor_parallel_size=2,  # Use 2 GPUs
    trust_remote_code=True
)

prompt = "[INST] Write a Python function to calculate fibonacci numbers [/INST]"
sampling_params = SamplingParams(temperature=0.3, max_tokens=150)
output = llm.generate([prompt], sampling_params)[0]

print(f"Model: Mixtral-8x7B")
print(f"Response:\n{output.outputs[0].text}")

### Example 4: Loading Quantized Models

Quantization reduces model size and memory requirements.

In [ ]:
# AWQ Quantized Model (4-bit)
# Reduces memory by ~4x with minimal quality loss
llm = LLM(
    model="TheBloke/Llama-2-7B-AWQ",  # AWQ quantized version
    quantization="awq",
    trust_remote_code=True
)

print("Loaded AWQ quantized model")
print("Memory saved: ~75% (16GB -> 4GB)")

# Test generation
prompt = "The benefits of quantization are:"
sampling_params = SamplingParams(temperature=0.7, max_tokens=60)
output = llm.generate([prompt], sampling_params)[0]
print(f"\nResponse: {output.outputs[0].text}")

---

## Advanced LLM Configuration

Understanding all the parameters for initializing an LLM.

In [ ]:
from vllm import LLM

llm = LLM(
    # === Model Configuration ===
    model="meta-llama/Llama-2-7b-hf",  # HuggingFace model ID or local path
    tokenizer="meta-llama/Llama-2-7b-hf",  # Optional: separate tokenizer
    tokenizer_mode="auto",  # "auto", "slow", or "mistral"
    trust_remote_code=True,  # Allow custom code in model

    # === GPU Configuration ===
    tensor_parallel_size=1,  # Number of GPUs for tensor parallelism
    pipeline_parallel_size=1,  # Number of pipeline stages (experimental)
    gpu_memory_utilization=0.90,  # Fraction of GPU memory to use (0.0-1.0)

    # === Memory and Sequence Configuration ===
    max_model_len=4096,  # Maximum sequence length
    max_num_seqs=256,  # Maximum concurrent sequences
    block_size=16,  # KV cache block size (PagedAttention)
    swap_space=4,  # CPU swap space in GiB (default: 4)

    # === Quantization ===
    quantization=None,  # None, "awq", "gptq", "squeezellm", "fp8"

    # === Data Types ===
    dtype="auto",  # "auto", "half" (FP16), "float16", "bfloat16", "float32"

    # === Optimization ===
    enforce_eager=False,  # Disable CUDA graph (for debugging)
    disable_custom_all_reduce=False,  # Disable custom all-reduce kernel

    # === Other ===
    seed=0,  # Random seed for reproducibility
    revision=None,  # HuggingFace model revision/branch
    download_dir=None,  # Directory to download model
)

print("LLM initialized with advanced configuration")

### Key Parameters Explained

In [ ]:
#### 1. tensor_parallel_size
Splits model across multiple GPUs **within a single node**
# Example: 70B model on 4 GPUs
llm = LLM(model="meta-llama/Llama-2-70b", tensor_parallel_size=4)

In [ ]:
#### 2. gpu_memory_utilization
Controls how much GPU memory vLLM can use
- Higher = More concurrent requests, but risk of OOM
- Lower = Safer, but fewer concurrent requests
# Conservative (safer)
llm = LLM(model="...", gpu_memory_utilization=0.85)
# Aggressive (maximum throughput)
llm = LLM(model="...", gpu_memory_utilization=0.95)

In [ ]:
#### 3. max_model_len
Maximum total sequence length (prompt + generation)
# For long documents
llm = LLM(model="...", max_model_len=8192)
# For short conversations (saves memory)
llm = LLM(model="...", max_model_len=2048)

In [ ]:
#### 4. max_num_seqs
Maximum number of sequences processed concurrently
# High throughput server
llm = LLM(model="...", max_num_seqs=512)
# Low latency server
llm = LLM(model="...", max_num_seqs=64)

#### 5. swap_space

CPU swap space in GiB for offloading KV cache blocks from GPU to CPU memory

**Purpose:**

- Acts as "overflow" storage when GPU memory is full

- Prevents Out-of-Memory (OOM) errors during traffic spikes

- Swaps inactive KV cache blocks to CPU RAM temporarily

- Automatically swaps blocks back to GPU when needed

In [ ]:
**How it works:**
GPU Memory (High Priority):
├─ Model weights (always in GPU)
├─ Active KV cache blocks (currently processing)
└─ [FULL] ❌

CPU Swap Space (Overflow):
├─ Inactive KV cache blocks (waiting requests)
└─ Temporarily swapped blocks

In [ ]:
**Configuration Examples:**
# Minimal swap (default, low CPU RAM usage)
llm = LLM(model="...", swap_space=4)  # 4 GiB

# Medium swap (balanced)
llm = LLM(model="...", swap_space=8)  # 8 GiB

# Large swap (handle more concurrent requests)
llm = LLM(model="...", swap_space=16)  # 16 GiB

# No swap (disable swapping, may OOM on spikes)
llm = LLM(model="...", swap_space=0)  # Disabled

**When to increase swap_space:**

- ✅ High traffic variability (sudden request bursts)

- ✅ Many concurrent long-context requests

- ✅ Large max_num_seqs with limited GPU memory

- ✅ Want to avoid OOM errors in production

**When to decrease/disable swap_space:**

- ❌ Limited CPU RAM available

- ❌ Prefer failing fast over slower swapped requests

- ❌ Predictable, steady traffic patterns

- ❌ Latency-critical applications (swapping adds latency)

**Trade-offs:**

- **Pro:** Prevents OOM, handles traffic spikes gracefully

- **Con:** Swapped requests have higher latency (GPU↔CPU transfer overhead)

- **Con:** Uses CPU memory (reduce if CPU RAM is limited)

---

#### 6. enforce_eager

Controls whether to use **CUDA graphs** (optimized) or **eager execution** (step-by-step)

**Simple Analogy:**

Think of making coffee ☕:

- **CUDA Graphs (enforce_eager=False)**: Pre-record all steps once, then replay the recording super fast

  - "Record: grind → brew → pour" → Replay this recording for every cup

  - ✅ Faster (after initial recording)

  - ❌ Can't change steps easily

- **Eager Execution (enforce_eager=True)**: Execute each step individually every time

  - "Grind... now brew... now pour..." (step by step)

  - ✅ Flexible, easy to debug, see what's happening

  - ❌ Slower (repeats work)

**Technical Explanation:**

🚀 **CUDA Graphs (Default: enforce_eager=False)**

- Records GPU operations once, replays the graph repeatedly

- Reduces CPU→GPU communication overhead

- **10-20% faster** for inference

- Used in production for maximum performance

🐛 **Eager Execution (enforce_eager=True)**

- Executes each GPU operation individually

- Easier to debug (you can see each step)

- Better error messages

- Allows dynamic behavior (control flow changes)

**When to use enforce_eager=True (Enable Eager Mode):**

In [ ]:
✅ **Debugging issues:**
# Your model crashes with cryptic error
llm = LLM(
    model="...",
    enforce_eager=True  # Get better error messages
)

In [ ]:
✅ **Development/Testing:**
# Experimenting with new models or features
llm = LLM(
    model="custom-model",
    enforce_eager=True  # Easier to debug custom code
)

In [ ]:
✅ **Dynamic batch sizes or shapes:**
# If your requests have highly variable lengths
llm = LLM(
    model="...",
    enforce_eager=True  # CUDA graphs don't handle variability well
)

In [ ]:
✅ **Profiling/Inspection:**
# When you need to profile each operation
llm = LLM(
    model="...",
    enforce_eager=True  # See individual op timings
)

**When to use enforce_eager=False (Use CUDA Graphs - DEFAULT):**

In [ ]:
✅ **Production deployment:**
# Maximum performance for serving users
llm = LLM(
    model="...",
    enforce_eager=False  # Default, fastest inference
)

In [ ]:
✅ **Stable workloads:**
# Consistent request patterns
llm = LLM(
    model="...",
    enforce_eager=False  # Benefit from graph optimization
)

In [ ]:
✅ **Benchmarking:**
# Measuring peak performance
llm = LLM(
    model="...",
    enforce_eager=False  # Get best possible speed
)

**Example: Debugging with enforce_eager**

In [ ]:
# Scenario: Model crashes with unclear error

# Step 1: Try with CUDA graphs (default)
try:
    llm = LLM(model="problematic-model", enforce_eager=False)
    output = llm.generate(["test"])
except Exception as e:
    print(f"Error with CUDA graphs: {e}")
    # Error message might be vague: "CUDA error 700"

# Step 2: Enable eager mode for better diagnostics
llm = LLM(model="problematic-model", enforce_eager=True)
output = llm.generate(["test"])
# Now you get clear error:
# "ValueError: attention_mask has wrong shape at layer 12"
# Much easier to debug! 🎯

**Performance Comparison:**

In [ ]:
import time

# Test with CUDA graphs (default)
llm_fast = LLM(model="meta-llama/Llama-2-7b-hf", enforce_eager=False)
start = time.time()
for _ in range(100):
    llm_fast.generate(["Hello"], SamplingParams(max_tokens=10))
cuda_graph_time = time.time() - start

# Test with eager execution
llm_debug = LLM(model="meta-llama/Llama-2-7b-hf", enforce_eager=True)
start = time.time()
for _ in range(100):
    llm_debug.generate(["Hello"], SamplingParams(max_tokens=10))
eager_time = time.time() - start

print(f"CUDA Graphs: {cuda_graph_time:.2f}s")
print(f"Eager Mode: {eager_time:.2f}s")
print(f"Speedup: {eager_time/cuda_graph_time:.1f}x faster with CUDA graphs")
# Typical output:
# CUDA Graphs: 8.5s
# Eager Mode: 10.2s
# Speedup: 1.2x faster with CUDA graphs

**Real-World Decision Tree:**

In [ ]:
Are you in production?
├─ YES → enforce_eager=False (use CUDA graphs for speed)
└─ NO
   ├─ Is your model working correctly?
   │  ├─ YES → enforce_eager=False (optimize performance)
   │  └─ NO → enforce_eager=True (debug mode)
   └─ Are you developing/experimenting?
      └─ YES → enforce_eager=True (easier iteration)

**Summary:**

- **Default (enforce_eager=False)**: Use CUDA graphs → Faster, production-ready

- **Debug (enforce_eager=True)**: Step-by-step execution → Slower, better errors

- **Rule of thumb**: Use `True` for debugging, `False` for production

---

## Quantization: Reducing Model Size

Quantization reduces model precision to save memory and increase speed.

### Understanding Quantization

| Format | Bits | Memory (7B model) | Quality | Speed |

|--------|------|-------------------|---------|-------|

| FP32 | 32 | 28 GB | 100% | Baseline |

| FP16/BF16 | 16 | 14 GB | 99.9% | 1.2x faster |

| INT8 | 8 | 7 GB | 98-99% | 1.5x faster |

| AWQ/GPTQ | 4 | 3.5 GB | 95-98% | 1.8x faster |

| FP8 | 8 | 7 GB | 99% | 2x faster |

### Example 1: AWQ Quantization (Recommended)

**AWQ** (Activation-aware Weight Quantization) is the recommended 4-bit quantization method.

In [ ]:
# Load AWQ quantized model
llm_awq = LLM(
    model="TheBloke/Llama-2-7B-AWQ",
    quantization="awq",
    max_model_len=4096,
    trust_remote_code=True
)

prompt = "List 5 advantages of quantization:"
sampling_params = SamplingParams(temperature=0.7, max_tokens=100)
output = llm_awq.generate([prompt], sampling_params)[0]

print("AWQ Quantized Model (4-bit)")
print(f"Response: {output.outputs[0].text}")

### Example 2: GPTQ Quantization

In [ ]:
# Load GPTQ quantized model
llm_gptq = LLM(
    model="TheBloke/Llama-2-7B-GPTQ",
    quantization="gptq",
    trust_remote_code=True
)

print("GPTQ Quantized Model (4-bit) loaded successfully")

### When to Use Quantization?

✅ **Use quantization when**:

- Limited GPU memory

- Want to serve larger models on smaller GPUs

- Need higher throughput (more concurrent requests)

- Quality loss (2-5%) is acceptable

❌ **Avoid quantization when**:

- Maximum quality is critical

- Plenty of GPU memory available

- Running benchmarks or research

---

## Model Serving: OpenAI-Compatible API Server

vLLM provides a production-ready API server compatible with OpenAI's API format.

### Starting the API Server (Command Line)

Open a terminal and run:

In [ ]:
# Basic server
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-2-7b-hf \
    --port 8000

# Advanced configuration
python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-2-7b-hf \
    --host 0.0.0.0 \
    --port 8000 \
    --tensor-parallel-size 2 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.9 \
    --trust-remote-code

# With quantization
python -m vllm.entrypoints.openai.api_server \
    --model TheBloke/Llama-2-7B-AWQ \
    --quantization awq \
    --port 8000

### Server Configuration Options

In [ ]:
# Core Options
--model MODEL_NAME              # Model to serve
--host HOST                     # Server host (default: localhost)
--port PORT                     # Server port (default: 8000)

# GPU Configuration
--tensor-parallel-size N        # Number of GPUs
--gpu-memory-utilization FRAC  # GPU memory fraction (0.0-1.0)

# Model Configuration
--max-model-len LENGTH          # Max sequence length
--max-num-seqs N               # Max concurrent sequences
--quantization METHOD          # awq, gptq, squeezellm

# API Configuration
--api-key KEY                   # API key for authentication
--served-model-name NAME       # Model name in API responses
--chat-template PATH           # Custom chat template

# Logging
--log-level LEVEL              # debug, info, warning, error

### Using the API Server from Python

Once the server is running, you can use it with the OpenAI Python client.

In [ ]:
# Install OpenAI client if needed
# !pip install openai

from openai import OpenAI

# Initialize client pointing to vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",  # vLLM server address
    api_key="dummy-key"  # vLLM doesn't require real key by default
)

# Completion API (text generation)
response = client.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    prompt="The capital of France is",
    max_tokens=50,
    temperature=0.7
)

print("Completion API Response:")
print(response.choices[0].text)

In [ ]:
# Chat Completion API (conversational)
response = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is machine learning?"}
    ],
    max_tokens=100,
    temperature=0.7
)

print("Chat Completion API Response:")
print(response.choices[0].message.content)

### Streaming Responses

Get tokens as they're generated for better user experience.

In [ ]:
# Streaming completion
stream = client.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    prompt="Write a short poem about AI:",
    max_tokens=100,
    temperature=0.8,
    stream=True  # Enable streaming
)

print("Streaming response:")
for chunk in stream:
    if chunk.choices[0].text:
        print(chunk.choices[0].text, end='', flush=True)
print()  # New line at end

---

## Multi-GPU Deployment

Deploy large models across multiple GPUs using tensor parallelism.

### Tensor Parallelism Explained

**Tensor Parallelism** splits each layer of the model across multiple GPUs:

In [ ]:
Single GPU:
GPU 1: [Full Model - 70B params]
❌ Won't fit in memory!

Tensor Parallelism (4 GPUs):
GPU 1: [1/4 of each layer]
GPU 2: [1/4 of each layer]
GPU 3: [1/4 of each layer]
GPU 4: [1/4 of each layer]
✅ Each GPU handles 17.5B params

**When to use**:

- Model doesn't fit on single GPU

- All GPUs must be on same machine

- Fast inter-GPU communication (NVLink preferred)

In [ ]:
# Example: Load 70B model on 4 GPUs
llm_large = LLM(
    model="meta-llama/Llama-2-70b-hf",
    tensor_parallel_size=4,  # Use 4 GPUs
    max_model_len=4096,
    trust_remote_code=True
)

print("70B model loaded across 4 GPUs")
print("Each GPU handles ~17.5B parameters")

### GPU Memory Requirements

Estimate GPU memory needed for your model:

In [ ]:
# Formula:
# Memory (GB) = (Parameters × Bytes_per_param × 1.2) / GPUs

# Examples (FP16):
# 7B model:  14GB (1 GPU)
# 13B model: 26GB (1-2 GPUs)
# 70B model: 140GB (4-8 GPUs)

# With AWQ quantization (4-bit):
# 7B model:  4GB (1 GPU)
# 13B model: 7GB (1 GPU)
# 70B model: 35GB (2-4 GPUs)

---

## Model-Specific Prompt Formats

Different models expect different prompt formats, especially for instruct/chat models.

### LLaMA 2 Chat Format

In [ ]:
# LLaMA 2 Chat expects specific format
llm = LLM(model="meta-llama/Llama-2-7b-chat-hf")

# System + User prompt format
prompt = """<s>[INST] <<SYS>>
You are a helpful, respectful and honest assistant.
<</SYS>>

What is the capital of France? [/INST]"""

sampling_params = SamplingParams(temperature=0.1, max_tokens=50)
output = llm.generate([prompt], sampling_params)[0]
print(output.outputs[0].text)

### Mistral Instruct Format

In [ ]:
# Mistral Instruct format
llm = LLM(model="mistralai/Mistral-7B-Instruct-v0.3")

prompt = "[INST] Explain quantum computing in simple terms [/INST]"

sampling_params = SamplingParams(temperature=0.7, max_tokens=100)
output = llm.generate([prompt], sampling_params)[0]
print(output.outputs[0].text)

### Using Chat Templates (Recommended)

vLLM supports automatic chat template formatting.

In [ ]:
# When using the API server, messages are automatically formatted
# No need to manually format prompts!

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

response = client.chat.completions.create(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    messages=[
        {"role": "user", "content": "What is AI?"}
    ]
)
# vLLM automatically applies correct format: [INST] What is AI? [/INST]

---

## Monitoring and Debugging

Tools to monitor your vLLM deployment.

In [ ]:
import torch

# Check GPU memory usage
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"  Total memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
        print(f"  Allocated: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
        print(f"  Reserved: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB")
        print(f"  Free: {(torch.cuda.get_device_properties(i).total_memory - torch.cuda.memory_reserved(i)) / 1e9:.2f} GB")

In [ ]:
# Check vLLM version and configuration
import vllm
print(f"vLLM version: {vllm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

---

## Best Practices for Model Serving

In [ ]:
### 1. Choose the Right Model Size
# GPU Memory Guide:
# 8-10 GB  → 7B models (or 13B quantized)
# 16-24 GB → 13B models (or 30B quantized)
# 40-48 GB → 30-34B models (or 70B quantized)
# 80+ GB   → 70B+ models

In [ ]:
### 2. Optimize for Your Use Case
# High throughput (many users):
LLM(
    model="...",
    max_num_seqs=512,
    max_model_len=2048,
    gpu_memory_utilization=0.95
)

# Low latency (fast response):
LLM(
    model="...",
    max_num_seqs=32,
    max_model_len=2048,
    gpu_memory_utilization=0.85
)

# Long context (documents):
LLM(
    model="...",
    max_num_seqs=64,
    max_model_len=8192,
    gpu_memory_utilization=0.90
)

### 3. Use Quantization When Appropriate

- AWQ for best quality (4-bit)

- GPTQ as alternative (4-bit)

- Test quality before production

In [ ]:
### 4. Monitor Performance
# Watch GPU usage
watch -n 1 nvidia-smi

# Monitor vLLM logs
python -m vllm.entrypoints.openai.api_server ... --log-level info

---

## Summary

In this notebook, you learned:

- ✅ Comprehensive list of supported models (LLaMA, Mistral, Mixtral, etc.)

- ✅ How to load different model architectures

- ✅ Advanced LLM configuration parameters

- ✅ Quantization methods (AWQ, GPTQ) for memory efficiency

- ✅ Setting up OpenAI-compatible API server

- ✅ Multi-GPU deployment with tensor parallelism

- ✅ Model-specific prompt formats

- ✅ Monitoring and debugging tools

- ✅ Best practices for production deployment

### Next Steps

Continue to the next notebook to learn about:

- Automatic Prefix Caching (APC)

- How prefix caching works

- Implementation and best practices